In [3]:
import matplotlib.pyplot as plt

In [4]:
import os
import scanpy as sc

def process_sample(
    samples, 
    color_map, 
    gene_markers, 
    idir, 
    filt_h5ad_dir, 
    odir
):
    """
    Processes spatial transcriptomics data for a list of samples.

    Parameters:
        samples (list): List of sample names.
        color_map (dict): Dictionary mapping cell types to colors.
        gene_markers (list): List of gene markers for heatmap generation.
        idir (str): Input directory containing spatial data.
        odir (str): Output directory for saving results.
        create_tangram_scores_df (func): Function to generate tangram scores DataFrame.
    """
    os.makedirs(odir, exist_ok = True)
    for sample_nm in samples:
        print(f"Processing {sample_nm}...")
        
        # Define input and output file paths
        # idir for imputed
        # idir for spatial with predicted celltype
        sample_dir = os.path.join(idir, sample_nm)
        #adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

        filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
        ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
        patdir = os.path.join(odir, sample_nm)
        os.makedirs(patdir, exist_ok=True)

        # Read data
        #adge = sc.read_h5ad(adge_file)
        adata_pred_filt = sc.read_h5ad(ad_sp_file)

        
        # Subset the AnnData object
        #sc.pl.spatial(adata_pred_filt_cropped, color = 'pred_cell_types', size = 2)
        # Subsetting adge based on cells in adata_pred (should not have undetermined cells)
        subset_cells = adata_pred_filt.obs_names  # Extract cell names from the subset object
        # Subset the original AnnData object
        #adge_filt = adge[adge.obs_names.isin(subset_cells)].copy()
        
        # Subset and sync spatial data
        #adge_filt.obs['pred_cell_types'] = adata_pred_filt.obs['pred_cell_types']
        #adge_filt.obsm['spatial'] = adata_pred_filt.obsm['spatial']
        #adge_filt.uns['spatial'] = adata_pred_filt.uns['spatial']

        # Plot spatial data by cell type
        plot_spatial_by_celltype(adata_pred_filt, outdir=patdir, sample_nm=sample_nm, palette_dict=color_map)

        # Create expression heatmaps
        #create_expression_heatmaps(
        #    raw_adata=adata_pred_filt,
        #    imputed_adata=adge_filt,
        #    markers=gene_markers,
        #    groupby="pred_cell_types",
        #    outdir=patdir,
        #    sample_name=sample_nm
        #)

        # Plot n counts distribution
        #plot_n_counts_distribution(
        ##    adata=adata_pred_filt,
        #    output_dir=patdir,
        #    sample_name=sample_nm
        #)


In [5]:
def plot_spatial_by_celltype(adata, outdir, sample_nm, palette_dict):
    """
    Generate spatial plots for all cell types and individual cell types in an AnnData object.
    
    Parameters:
    -----------
    adata : AnnData
        The annotated data object containing spatial transcriptomics data.
    sb : str
        The output directory where the plots will be saved.
    sample_nm : str
        Sample name to include in the filenames.
    palette_dict : dict
        A dictionary mapping cell types to specific colors.
    """

    #sc.set_figure_params(dpi_save=3000) #set the dpi for the H&E
    sc.set_figure_params(dpi_save=500) #set the dpi for the H&E


    # Plotting expression only without H&E
    # Fixed cell type order based on palette_dict keys
    unique_subclusters = adata.obs['pred_cell_types'].unique()
    
    # Plot for all cell types combined
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')  # Set figure background to white
    ax.set_facecolor('black')  # Set axes background to white
    
    sc.pl.spatial(
        adata,
        alpha_img=0,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False
    )
    
    # Adjust legend text color to black
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_color('white')
    
    # Save the combined plot
    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue.pdf", bbox_inches='tight')
    plt.close(fig)

    # Plot for each individual cell type
    #for celltype in unique_subclusters:
        # Subset AnnData object for the current cell type
    #    celltype_subset = adata[adata.obs['pred_cell_types'] == celltype]

        # Create a new figure
    #    fig, ax = plt.subplots(figsize=(6, 6))
    #    fig.patch.set_facecolor('white')  # Set figure background to white
    #    ax.set_facecolor('white')  # Set axes background to white

        # Plot spatial data for the current cell type
    #    sc.pl.spatial(
    #        celltype_subset,
    #        img_key=None,
    #        palette=[palette_dict.get(celltype, '#000000')],
    #        color='pred_cell_types',
    #        ax=ax,
    #        size=2,
    #        show=False,
    #        legend_loc=None
    #    )

        # Save the individual cell type plot
    #    plt.savefig(f"{outdir}/{sample_nm}_{celltype}_spatial.pdf", bbox_inches='tight')
    #    plt.close(fig)  # Close the figure to save memory



    # Plot for all cell types combined minus hepatocytes
    adata_noheps = adata[~adata.obs['pred_cell_types'].isin(['Hepatocytes'])]
    unique_subclusters_noheps = adata.obs['pred_cell_types'].unique()

    
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')  # Set figure background to white
    ax.set_facecolor('black')  # Set axes background to white
    
    sc.pl.spatial(
        adata_noheps,
        alpha_img=0,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters_noheps)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False,
    )
    
    # Adjust legend text color to black
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_color('white')
    
    # Save the combined plot
    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue_noheps.pdf", bbox_inches='tight')
    plt.close(fig)


    # Plot for just H&E cropped
    print("Plotting H&E")
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('black')  # Set figure background to white
    ax.set_facecolor('black')  # Set axes background to white
    # 00FFFFFF
    #sc.pl.spatial(adata_noheps, palette=[neon_rainbow_colors_broad_transp.get(celltype, '#FFFFFF00') for celltype in sorted(unique_subclusters)],
    #            color='pred_cell_types', size = 2, show=False)
    sc.pl.spatial(
        adata_noheps,
        #img_key=None,
        palette=[palette_dict.get(celltype, '#000000') for celltype in sorted(unique_subclusters_noheps)],
        color='pred_cell_types',
        ax=ax,
        size=2,
        show=False,
        alpha = 0
        
    )
    plt.savefig(f"{outdir}/{sample_nm}_anno_broad_whole_tissue_he.pdf", bbox_inches='tight')
    plt.close(fig)


In [6]:
# from paper very original
neon_rainbow_colors_broad = {
        'Hepatocytes': '#FF007F',    # Neon pink #maroon is '#800000'
        'Myeloid': '#0000FF',           # Neon blue
        'T_NK': '#FFFA00',              # Neon yellow
        'Cholangiocyte': '#00FF00',  # Neon green
        'HSC': '#00FFFF',            # Neon cyan
        'Mast': '#FF1900',        # Neon red
        'Endothelial': '#8A2BE2',    # Neon purple
        'Schwann': '#FF1493',        # Neon deep pink
        'NK': '#FFFDBB',             # lighter yellow
        'B': '#FF8800'               # Neon hot pink
    }

In [7]:
# hepatocytes picked from paper
# hepatocytes - #ec2d4b
# chol - #5baf5c
# endo - #633d99
# hsc - #75c2e0
# myeloid - #474d9c
# t /nk #f6ed24
# b #f78a1f

In [8]:
  brin_markers = [
        
        "cd19", "ms4a1",
        "krt19", "fxyd2", "spp1", 
        "epcam", "sox9", "anxa4", "sry", "krt1",
        "pecam1", "sele", "flt4", 
        "lyve1", "mcam", "cd34", "ptprc", "stab2" , "ptprb",    # added endothelial
        "col1a1", "fap", "adamts13",
        "ngfr", "cygb", "hgf", "rbp1", #added stellate cells
        "msln", "thy1",  #fibroblasts
        "grem1", "aspn", "calca", "eln", # Gremlin1, Asporin, calcitonin a, Elastin #added fibroblasts
        "msln", #not found in imputated
        "cyp2e1", "hnf4a", "crp", "alb", 
        "serpina1", "ttr", #added heps
        "c1qa","cd163", "timd4",
        "cd68", "ms4a7", # added myeloid
        "cd69", "trbc2", "cd3d"
    
    
    ]


In [9]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'
filt_h5ad_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/tangram_scores_dist2/'
odir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/misc/qc_plots_whole_image_broad_ct_darkbg/'

In [10]:
samples = [ 'HL160029_filt', 'HL230324_filt','HL20221019_broad_no_mast_filt', 'HL170058_filt']

In [11]:
process_sample(
    samples, 
    neon_rainbow_colors_broad, 
    brin_markers, 
    idir, 
    filt_h5ad_dir,
    odir)

Processing HL160029_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL230324_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL20221019_broad_no_mast_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
Processing HL170058_filt...


/tscc/nfs/home/cmiciano/miniconda3/envs/spatial/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[value_to_plot + "_colors"] = colors_list


Plotting H&E
